In [ ]:
import pandas as pd

In [ ]:
playstore_apps = pd.read_csv("/content/drive/MyDrive/CSVs/googleplaystore.csv")

In [ ]:
playstore_reviews = pd.read_csv("/content/drive/MyDrive/CSVs/googleplaystore_user_reviews.csv")

In [ ]:
playstore_reviews["Sentiment_Polarity_abs"] = playstore_reviews["Sentiment_Polarity"].abs()
playstore_reviews_polarity_mean = playstore_reviews.groupby("App").agg({"Sentiment_Polarity": ["mean"], "Sentiment_Polarity_abs": ["max"]}).reset_index()
playstore_reviews_polarity_mean.columns = ["App", "Sentiment_Polarity_mean", "Sentiment_Polarity_max"]

In [ ]:
playstore_apps.duplicated().value_counts()

False    10358
True       483
dtype: int64

In [ ]:
playstore_apps.duplicated(subset="App").value_counts()

False    9660
True     1181
dtype: int64

In [ ]:
playstore_apps[~playstore_apps.duplicated(subset="App")]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3,9.5M,"1,000+",Free,0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [ ]:
pd.merge(playstore_reviews_polarity_mean, playstore_apps[~playstore_apps.duplicated(subset="App")][["App", "Type"]]).duplicated().value_counts()

False    1020
dtype: int64

In [ ]:
pd.merge(playstore_reviews_polarity_mean, playstore_apps[~playstore_apps.duplicated(subset="App")][["App", "Type"]]).duplicated(subset="App").value_counts()

False    1020
dtype: int64

In [ ]:
playstore_merge = pd.merge(playstore_reviews_polarity_mean, playstore_apps[~playstore_apps.duplicated(subset="App")][["App", "Type", "Rating"]])
playstore_merge

,App,Sentiment_Polarity_mean,Sentiment_Polarity_max,Type,Rating
0,10 Best Foods for You,0.470733,1.000000,Free,4.0
1,11st,0.181294,1.000000,Free,3.8
2,1800 Contacts - Lens Store,0.318145,0.838542,Free,4.7
3,1LINE – One Line with One Touch,0.196290,1.000000,Free,4.6
4,2018Emoji Keyboard 😂 Emoticons Lite -sticker&gif,0.449566,1.000000,Free,4.2
...,...,...,...,...,...
1015,Hotspot Shield Free VPN Proxy & Wi-Fi Security,0.251765,1.000000,Free,4.2
1016,Hotstar,0.038178,1.000000,Free,4.3
1017,Hotwire Hotel & Car Rental App,0.187029,0.875000,Free,4.3
1018,Housing-Real Estate & Property,-0.021427,0.800000,Free,4.1


In [ ]:
playstore_merge["Sentiment_Polarity_mean"] = playstore_merge["Sentiment_Polarity_mean"].abs()
playstore_merge

,App,Sentiment_Polarity_mean,Sentiment_Polarity_max,Type,Rating
0,10 Best Foods for You,0.470733,1.000000,Free,4.0
1,11st,0.181294,1.000000,Free,3.8
2,1800 Contacts - Lens Store,0.318145,0.838542,Free,4.7
3,1LINE – One Line with One Touch,0.196290,1.000000,Free,4.6
4,2018Emoji Keyboard 😂 Emoticons Lite -sticker&gif,0.449566,1.000000,Free,4.2
...,...,...,...,...,...
1015,Hotspot Shield Free VPN Proxy & Wi-Fi Security,0.251765,1.000000,Free,4.2
1016,Hotstar,0.038178,1.000000,Free,4.3
1017,Hotwire Hotel & Car Rental App,0.187029,0.875000,Free,4.3
1018,Housing-Real Estate & Property,0.021427,0.800000,Free,4.1


In [ ]:
promedios_type = playstore_merge.groupby("Type")["Sentiment_Polarity_mean"].mean()
promedios_type

Type
Free    0.212255
Paid    0.214263
Name: Sentiment_Polarity_mean, dtype: float64

In [ ]:
promedios_rating_free = playstore_merge[(playstore_merge["Type"] == "Free") & (playstore_merge["Sentiment_Polarity_max"] > 2*promedios_type["Free"])]["Rating"].mean()
promedios_rating_free

4.276259946949603

In [ ]:
promedios_rating_paid = playstore_merge[(playstore_merge["Type"] == "Paid") & (playstore_merge["Sentiment_Polarity_max"] > 2*promedios_type["Paid"])]["Rating"].mean()
promedios_rating_paid

4.3125

In [ ]:
promedios_rating_por_type = promedios_type.reset_index()
promedios_rating_por_type["Rating_mean"] = [promedios_rating_free, promedios_rating_paid]
promedios_rating_por_type

,Type,Sentiment_Polarity_mean,Rating_mean
0,Free,0.212255,4.27626
1,Paid,0.214263,4.31250
